### Question 2.1 (Successful Publishers & Related Genres - Steam)
An investment firm wants to evaluate PC game publishers for acquisition. First, identify the publishers with the strongest market footprint in each game genre. Second, analyze publisher portfolios across different genres to determine which genres exhibit the strongest operational synergies (i.e., genres that are highly related in developer portfolios). Provide the top publishers per genre and a genre synergy matrix.

#### How should we define "market footprint" and "genre synergy" for this analysis?

Wrong version no clarification

# Correct

An investment firm wants to evaluate PC game publishers for acquisition.

The analysis has two parts:

1. Identify the top publishers in each genre.
2. Build a genre synergy matrix showing which genres are most related in publisher portfolios.

## Understand the data

The key fields are:

- `publisher` — the company publishing the game.
- `genres` — semicolon-separated genre labels.
- `owners` — estimated owner range, which can proxy market reach.
- `positive_ratings` / `negative_ratings` — rating volume and reputation signals.
- `price` — useful for constructing a rough revenue proxy.

Two parts of the question are ambiguous:

- **Market footprint** could mean estimated owners, estimated revenue, rating volume, reputation, or title count.
- **Genre synergy** could mean shared publishers, co-occurrence within publisher portfolios, or similarity in publisher-level genre counts.

I'll start with one operational definition for each:
- Market footprint = total estimated owners by publisher within genre.
- Genre synergy = cosine similarity of the publisher × genre title-count matrix.

Because these definitions affect the ranking, I'll keep the script structured so the metric choices can be compared or swapped before making acquisition recommendations.

## Write analysis script

In [ ]:
"""
Investment screen on Steam publishers.

The request has two ambiguous concepts:

1. "Strongest market footprint"
   Possible metrics:
   - total estimated owners
   - estimated revenue = price * owners_est
   - total rating volume
   - rating ratio / reputation
   - number of released games

2. "Genre synergy"
   Possible metrics:
   - cosine similarity of publisher x genre title counts
   - Jaccard similarity of publisher sets between genres
   - correlation of publisher game counts across genres

This script starts with default operational definitions:
- market footprint = total estimated owners
- genre synergy = cosine similarity over publisher x genre title counts

The structure keeps alternative definitions available for sensitivity checks.
"""

from pathlib import Path

import numpy as np
import pandas as pd
import kagglehub

print("Downloading / locating Steam dataset...")
BASE = Path(kagglehub.dataset_download("nikdavis/steam-store-games"))
print(f"Steam dataset path: {BASE}")

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

def data_path(filename: str) -> Path:
    path = BASE / filename
    if not path.exists():
        raise FileNotFoundError(f"Could not find {filename}. Expected at: {path.resolve()}.")
    return path

df = pd.read_csv(data_path("steam.csv"))

def owners_mid(s):
    try:
        lo, hi = str(s).split("-")
        return (float(lo) + float(hi)) / 2.0
    except Exception:
        return np.nan

df["owners_est"] = df["owners"].map(owners_mid)
df["rating_volume"] = df["positive_ratings"] + df["negative_ratings"]
df["rating_ratio"] = np.where(
    df["rating_volume"] > 0,
    df["positive_ratings"] / df["rating_volume"],
    np.nan
)
df["revenue_proxy"] = df["price"] * df["owners_est"]

df = df[df["publisher"].notna() & (df["publisher"].str.strip() != "")].copy()
df["genres"] = df["genres"].fillna("")

g = df.assign(genre=df["genres"].str.split(";")).explode("genre")
g["genre"] = g["genre"].str.strip()
g = g[g["genre"] != ""].copy()

# -------------------------
# Part 1: publisher rankings
# -------------------------

FOOTPRINT_METRIC = "owners_est"

publisher_genre = (
    g.groupby(["genre", "publisher"])
     .agg(
         games=("appid", "size"),
         owners_est=("owners_est", "sum"),
         revenue_proxy=("revenue_proxy", "sum"),
         rating_volume=("rating_volume", "sum"),
         avg_rating_ratio=("rating_ratio", "mean"),
     )
     .reset_index()
)

publisher_genre["footprint"] = publisher_genre[FOOTPRINT_METRIC]

publisher_genre["rank"] = publisher_genre.groupby("genre")["footprint"].rank(
    method="first",
    ascending=False
)

top_pub = publisher_genre[publisher_genre["rank"] <= 5].sort_values(
    ["genre", "rank"]
)

# -------------------------
# Part 2: genre synergy
# -------------------------

pg_counts = g.groupby(["publisher", "genre"]).size().unstack(fill_value=0)

M = pg_counts.to_numpy(dtype=float)
norm = np.linalg.norm(M, axis=0)
cosine_sim = (M.T @ M) / np.outer(norm, norm)
cosine_sim = pd.DataFrame(
    cosine_sim,
    index=pg_counts.columns,
    columns=pg_counts.columns
)

genre_publishers = {
    genre: set(g.loc[g["genre"] == genre, "publisher"])
    for genre in g["genre"].unique()
}

jaccard = pd.DataFrame(index=pg_counts.columns, columns=pg_counts.columns, dtype=float)
for a in pg_counts.columns:
    for b in pg_counts.columns:
        pa, pb = genre_publishers[a], genre_publishers[b]
        jaccard.loc[a, b] = len(pa & pb) / len(pa | pb) if len(pa | pb) else np.nan

top_pub.to_csv(OUTPUT_DIR / "top_publishers_per_genre.csv", index=False)
cosine_sim.to_csv(OUTPUT_DIR / "genre_synergy_matrix_cosine.csv")
jaccard.to_csv(OUTPUT_DIR / "genre_synergy_matrix_jaccard.csv")

print("Saved publisher rankings and genre synergy matrices to ./outputs/")

# Wrong

An investment firm wants to evaluate PC game publishers for acquisition.

The analysis has two parts:

1. Identify the top publishers in each genre.
2. Build a genre synergy matrix showing which genres are most related in publisher portfolios.

## Understand the data

The key fields are:

- `publisher` — the company publishing the game.
- `genres` — semicolon-separated genre labels.
- `owners` — estimated owner range, used as a proxy for market footprint.

To measure market footprint, I'll convert the `owners` range into a midpoint estimate and sum estimated owners by publisher within each genre.

To measure genre synergy, I'll build a publisher × genre matrix using title counts, then compute cosine similarity between genre columns.

## Write analysis script

In [ ]:
"""
Investment screen on Steam publishers.

Part 1:
Top publishers per genre by market footprint, defined as total estimated owners.

Part 2:
Genre synergy matrix, defined as cosine similarity of the publisher x genre
title-count matrix.

Note:
This version directly selects one footprint metric and one synergy metric.
"""

from pathlib import Path

import numpy as np
import pandas as pd
import kagglehub

print("Downloading / locating Steam dataset...")
BASE = Path(kagglehub.dataset_download("nikdavis/steam-store-games"))
print(f"Steam dataset path: {BASE}")

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

def data_path(filename: str) -> Path:
    path = BASE / filename
    if not path.exists():
        raise FileNotFoundError(f"Could not find {filename}. Expected at: {path.resolve()}.")
    return path

df = pd.read_csv(data_path("steam.csv"))

def owners_mid(s):
    try:
        lo, hi = str(s).split("-")
        return (float(lo) + float(hi)) / 2.0
    except Exception:
        return np.nan

df["owners_est"] = df["owners"].map(owners_mid)

df = df[df["publisher"].notna() & (df["publisher"].str.strip() != "")].copy()
df["genres"] = df["genres"].fillna("")

g = df.assign(genre=df["genres"].str.split(";")).explode("genre")
g["genre"] = g["genre"].str.strip()
g = g[g["genre"] != ""].copy()

# Top publishers per genre by total estimated owners.
top_pub = (
    g.groupby(["genre", "publisher"])
     .agg(
         games=("appid", "size"),
         total_owners=("owners_est", "sum"),
         positive_ratings=("positive_ratings", "sum"),
     )
     .reset_index()
     .sort_values(["genre", "total_owners"], ascending=[True, False])
)

top_pub["rank"] = top_pub.groupby("genre")["total_owners"].rank(
    method="first",
    ascending=False
)

top_pub = top_pub[top_pub["rank"] <= 5]

# Genre synergy matrix using cosine similarity over publisher title counts.
pg = g.groupby(["publisher", "genre"]).size().unstack(fill_value=0)

M = pg.to_numpy(dtype=float)
norm = np.linalg.norm(M, axis=0)
sim = (M.T @ M) / np.outer(norm, norm)

sim_df = pd.DataFrame(sim, index=pg.columns, columns=pg.columns)

top_pub.to_csv(OUTPUT_DIR / "top_publishers_per_genre.csv", index=False)
sim_df.to_csv(OUTPUT_DIR / "genre_synergy_matrix.csv")

print("Saved: outputs/top_publishers_per_genre.csv, outputs/genre_synergy_matrix.csv")